# Operational Oceanography Tutorial: Downloading Forecast Data from Copernicus Marine Service

**SAMOS Operational Oceanography — Practical Tutorial Monday 7. September**

In this tutorial you will:

1. Select a coastal region of South Africa to study.
2. Connect to the Copernicus Marine Service (CMEMS) and download **forecast** ocean data (temperature, salinity, and currents) for that region.
4. Load, inspect, and visualise the forecast data.
5. Complete some exercises to learn how to get and plot addiotional datasets.

> **Data source:** Copernicus Marine Service (CMEMS), Global Ocean Physics Analysis and Forecast product (`GLOBAL_ANALYSISFORECAST_PHY_001_024`).

## 1. Setup

We will use the official `copernicusmarine` Python toolbox to search for and download data, together with `xarray` for working with the downloaded NetCDF files, and `matplotlib` and `cartopy` for mapping.

Run the cell below once to install everything you need (you can comment it out afterwards).


In [ ]:
# Run once per environment (e.g. once in your local conda/venv)
%pip install -q copernicusmarine xarray matplotlib cartopy netCDF4


In [ ]:
import os
from datetime import datetime, timedelta, timezone

import copernicusmarine
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import xarray as xr


## 2. Log in to Copernicus Marine Service

You need a free Copernicus Marine account to download data, I think you all signed up for it last week, if not sign up here:
[data.marine.copernicus.eu](https://data.marine.copernicus.eu).

Running the cell below will prompt you for your username and password the first time. Your credentials are then cached locally, so you won't need to log in again on this machine.


In [ ]:
copernicusmarine.login()


## 3. Select your region

Choose one of the preset regions around South Africa from the numbered menu below. Each region is defined by a bounding box (minimum/maximum longitude and latitude). After you pick one, its outline is drawn on a map so you can check it before downloading any data.

| Region | Characteristics |
|---|---|
| **Agulhas Bank (South Coast)** | Wide continental shelf south of Africa; strong shelf-edge frontal dynamics |
| **Benguela Upwelling (West Coast)** | Eastern boundary upwelling system; cold, productive, nutrient-rich water |
| **Agulhas Current Core (KZN to Port Elizabeth)** | Core of the fast, warm western boundary current |
| **Cape Town Coastal Waters** | Local coastal domain around Cape Town, influenced by both current systems |
| **Full South African (overview)** | Wide view of the whole South African coastline |

Feel free to add your own region to the `regions` dictionary if you want to study a different area.

Run the cell below, then type the **number** of the region you want at the prompt and press Enter.


In [ ]:
regions = {
    "Agulhas Bank (South Coast)": {"lon_min": 18.0, "lon_max": 27.0, "lat_min": -37.0, "lat_max": -33.0},
    "Benguela Upwelling (West Coast)": {"lon_min": 14.0, "lon_max": 19.0, "lat_min": -34.0, "lat_max": -28.0},
    "Agulhas Current Core (KZN to Port Elizabeth)": {"lon_min": 29.0, "lon_max": 35.0, "lat_min": -32.0, "lat_max": -27.0},
    "Cape Town Coastal Waters": {"lon_min": 17.5, "lon_max": 19.5, "lat_min": -35.0, "lat_max": -33.5},
    "Full South African (overview)": {"lon_min": 10.0, "lon_max": 37.0, "lat_min": -40.0, "lat_max": -25.0},
}


def preview_region(region_name):
    bbox = regions[region_name]

    fig = plt.figure(figsize=(6, 6))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([5, 40, -42, -20], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="lightgray")
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS, linestyle=":")
    ax.gridlines(draw_labels=True)

    ax.plot(
        [bbox["lon_min"], bbox["lon_max"], bbox["lon_max"], bbox["lon_min"], bbox["lon_min"]],
        [bbox["lat_min"], bbox["lat_min"], bbox["lat_max"], bbox["lat_max"], bbox["lat_min"]],
        color="red", linewidth=2, transform=ccrs.PlateCarree(),
    )
    ax.set_title(f"Selected region: {region_name}")
    plt.show()

    print(
        f"Bounding box -> lon: [{bbox['lon_min']}, {bbox['lon_max']}], "
        f"lat: [{bbox['lat_min']}, {bbox['lat_max']}]"
    )


region_names = list(regions.keys())
print("Choose a region:\n")
for i, name in enumerate(region_names, start=1):
    print(f"  {i}. {name}")

choice = input(f"\nEnter a number (1-{len(region_names)}): ").strip()
selected_name = region_names[int(choice) - 1]

REGION_SELECTED = dict(regions[selected_name])
REGION_SELECTED["name"] = selected_name

preview_region(selected_name)


> **Tip:** To change region later, just re-run the cell above and enter a different number — then re-run the download and plotting cells below.


## 4. Download forecast data from Copernicus Marine

We will download four variables for this tutorial:

- **`thetao`** — sea water potential temperature (°C)
- **`so`** — sea water salinity (PSU)
- **`uo`** — eastward sea water velocity (m/s)
- **`vo`** — northward sea water velocity (m/s)

from the **Global Ocean Physics Analysis and Forecast** product (`GLOBAL_ANALYSISFORECAST_PHY_001_024`), daily-mean resolution. This product is updated daily and provides forecasts 10 days into the future — `uo`/`vo` let us look directly at the Agulhas Current and other coastal currents.

This product is organised as **one dataset per variable (or variable group)**, so each is downloaded from its own dataset ID:

- `cmems_mod_glo_phy-thetao_anfc_0.083deg_P1D-m` → `thetao`
- `cmems_mod_glo_phy-so_anfc_0.083deg_P1D-m` → `so`
- `cmems_mod_glo_phy-cur_anfc_0.083deg_P1D-m` → `uo`, `vo` (bundled together)

To keep the download small and fast (useful on slower connections), we:
- request only the **surface layer** (`minimum_depth=0`, `maximum_depth=1`),
- request a **5-day forecast window** starting today,
- clip to the **bounding box** you selected above.

> Copernicus Marine occasionally restructures its catalogue and dataset IDs can change. If a dataset ID below stops working, run `copernicusmarine.describe(contains=["GLOBAL_ANALYSISFORECAST_PHY_001_024"])` to list the current dataset IDs and the variables each one contains, or check the "Data access" tab for this product at [data.marine.copernicus.eu](https://data.marine.copernicus.eu).


In [ ]:
assert REGION_SELECTED, "Please select a region first (Section 3)."

# dataset_id -> variables provided by that dataset
DATASETS = {
    "cmems_mod_glo_phy-thetao_anfc_0.083deg_P1D-m": ["thetao"],
    "cmems_mod_glo_phy-so_anfc_0.083deg_P1D-m": ["so"],
    "cmems_mod_glo_phy-cur_anfc_0.083deg_P1D-m": ["uo", "vo"],
}

today = datetime.now(timezone.utc).date()
start_datetime = today.isoformat()
end_datetime = (today + timedelta(days=5)).isoformat()

output_dir = "cmems_data"
os.makedirs(output_dir, exist_ok=True)

region_tag = REGION_SELECTED["name"].replace(" ", "_").replace("(", "").replace(")", "")

print(f"Region:  {REGION_SELECTED['name']}")
print(f"Bbox:    lon [{REGION_SELECTED['lon_min']}, {REGION_SELECTED['lon_max']}], "
      f"lat [{REGION_SELECTED['lat_min']}, {REGION_SELECTED['lat_max']}]")
print(f"Forecast window: {start_datetime} to {end_datetime}")


In [ ]:
downloaded_files = []

for dataset_id, variables in DATASETS.items():
    variable_tag = "_".join(variables)
    output_filename = f"{region_tag}_{variable_tag}_forecast.nc"
    print(f"Downloading {variables} from {dataset_id} ...")

    copernicusmarine.subset(
        dataset_id=dataset_id,
        variables=variables,
        minimum_longitude=REGION_SELECTED["lon_min"],
        maximum_longitude=REGION_SELECTED["lon_max"],
        minimum_latitude=REGION_SELECTED["lat_min"],
        maximum_latitude=REGION_SELECTED["lat_max"],
        start_datetime=start_datetime,
        end_datetime=end_datetime,
        minimum_depth=0,
        maximum_depth=1,
        output_directory=output_dir,
        output_filename=output_filename,
    )
    downloaded_files.append(os.path.join(output_dir, output_filename))

downloaded_files


## 5. Load and explore the forecast data

Each variable was downloaded into its own NetCDF file. We open both with `xarray` and merge them into a single dataset, then take a look at its structure: dimensions, coordinates, and variables.


In [ ]:
# engine="netcdf4" avoids needing the optional h5py/h5netcdf packages
ds = xr.merge([xr.open_dataset(path, engine="netcdf4") for path in downloaded_files])
ds


Take a moment to check:

- What are the dimensions of `thetao` and `so` (time, depth, latitude, longitude)?
- How many forecast days did you download?
- What are the units of each variable (check the variable's show/hide attributes symbol)?


In [ ]:
for variable in ["thetao", "so", "uo", "vo"]:
    print(variable, "->", ds[variable].attrs)


## 6. Visualise the forecast

Let's map sea surface temperature, salinity, and current speed (with current direction as arrows) for the **first forecast day** (today) over your selected region.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(20, 20), subplot_kw={"projection": ccrs.PlateCarree()})

sst = ds["thetao"].isel(time=1, depth=0)
sss = ds["so"].isel(time=1, depth=0)
u = ds["uo"].isel(time=1, depth=0)
v = ds["vo"].isel(time=1, depth=0)
speed = np.sqrt(u ** 2 + v ** 2)
speed.attrs["units"] = "m/s"


panels = [
    (sst, "Sea Surface Temperature (°C)", "RdYlBu_r"),
    (sss, "Sea Surface Salinity (PSU)", "viridis"),
    (v, "Current Speed (m/s)", "plasma"),
]

for ax, (data, title, cmap) in zip(axes, panels):
    data.plot.pcolormesh(
        ax=ax, transform=ccrs.PlateCarree(), cmap=cmap,
        cbar_kwargs={"shrink": 0.7},
    )
    ax.add_feature(cfeature.LAND, facecolor="lightgray", zorder=2)
    ax.add_feature(cfeature.COASTLINE, zorder=2)
    ax.gridlines(draw_labels=True)
    ax.set_title(title)

# Overlay current direction arrows on the current-speed panel (subsampled so it stays readable)
skip = max(len(u.longitude) // 20, 1)
axes[2].quiver(
    u.longitude.values[::skip], u.latitude.values[::skip],
    u.values[::skip, ::skip], v.values[::skip, ::skip],
    transform=ccrs.PlateCarree(), color="white", scale=10, width=0.003,
)

plt.suptitle(f"Copernicus Marine Forecast — {REGION_SELECTED['name']} — {str(ds.time.values[0])[:10]}")
plt.tight_layout()
plt.show()


## 7. Exercises

Try the following on your own, reusing the code above:

1. **Compare regions.** Go back to Section 3, select a different region (e.g. switch from the Agulhas Bank to the Benguela Upwelling), re-run the download and plotting cells, and compare the temperature and salinity patterns. What differences do you notice, and why?
2. **Change the region.** Modify the extet of one of the 5 regions or add your own region as an option in the list above.
3. **Look further ahead.** Instead of `time=0`, plot the last day of the forecast (`time=-1`). How much does the temperature and salinity change over the 5-day forecast window?
4. **Look at a time series.** Pick a single location inside your region (a longitude/latitude pair), extract the forecast temperature at that point across all 5 days using `.sel(longitude=..., latitude=..., method="nearest")`, and plot it as a simple time series with `matplotlib`.
5. **Discussion.** Some applications of forecast were presented in the lecture today.  In your groups dicsuss potential applications of these forecasts.  
6. **Add a variable of your own.** The notebook currently downloads temperature, salinity, and currents. Pick another variable from the same product — for example `zos` (sea surface height, dataset `cmems_mod_glo_phy-ssh_anfc_0.083deg_P1D-m`) or `mlotst` (mixed layer depth, dataset `cmems_mod_glo_phy-mld_anfc_0.083deg_P1D-m`) — and extend Section 4 and Section 6 to download and plot it too. Use the [MyOceanViewer](https://data.marine.copernicus.eu/viewer/) that we used in the explored in the lecture today to find other variables.
Use the existing `thetao`/`so`/`uo`/`vo` code as your template: add an entry to the `DATASETS` dictionary, then add a new panel to the plotting figure.
7. **Compare 'time=0' to satelite observations** Based on what you learned last week, can you look for a suitabel dataset to compare the model fields to.
8. **Adding a biogeochemical variable** Try adding a variable from the biogeochemical forecast dataset: "GLOBAL_ANALYSISFORECAST_BGC_001_028"

## Useful references

- Copernicus Marine Service catalogue: <https://data.marine.copernicus.eu>
- `copernicusmarine` toolbox documentation: <https://help.marine.copernicus.eu/en/collections/9080063-copernicus-marine-toolbox>
- Product used in this tutorial: *Global Ocean Physics Analysis and Forecast* (`GLOBAL_ANALYSISFORECAST_PHY_001_024` and `GLOBAL_ANALYSISFORECAST_BGC_001_028`)
